# Lasso regression (L1 regularization)

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
Lasso also penalizes large coefficients, but with the **absolute value** instead of the square:

$$J(\mathbf w)=\frac{1}{2n}\sum_i(y_i-\hat y_i)^2+\alpha\sum_j\vert w_j\vert$$

This small change has a big effect: Lasso sets many coefficients to **exactly zero**. It drops useless features by itself: **automatic feature selection**.

## Why exactly zero? (the geometric picture)
The solution is where the error contours (orange ellipses) first touch the allowed region of coefficients. The L2 region is a circle, and the touch point usually has both coefficients non-zero. The L1 region is a diamond whose **corners lie on the axes**; the touch often happens at a corner, where one coefficient is zero.

> [Left: Ridge (circle). Right: Lasso (diamond), which sets w1 to zero at a corner.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/l1_l2.png)

*Left: Ridge (circle). Right: Lasso (diamond), which sets w1 to zero at a corner.*

## How is it solved? (coordinate descent)
The absolute value has no derivative at zero, so there is no closed formula. scikit-learn optimizes one coefficient at a time using **soft thresholding**:

$$w_j\leftarrow S(\rho_j,\alpha)=\text{sign}(\rho_j)\max(\vert\rho_j\vert-\alpha,0)$$

where $\rho_j$ is the correlation of feature $j$ with the current residual. If $\vert\rho_j\vert\le\alpha$, the coefficient becomes exactly zero.

| $\rho_j$ | $\alpha$ | New coefficient |
|---|---|---|
| 3.0 | 1.0 | 2.0 |
| 0.6 | 1.0 | 0 |
| −2.5 | 1.0 | −1.5 |

## The coefficient path
As $\alpha$ grows, coefficients drop to zero **one by one**. The last survivors are the most important features:

> [Lasso path on the diabetes data.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/lasso_path.png)

*Lasso path on the diabetes data.*

[[lab:reg_lasso]]

In the lab, raise $\alpha$ and follow the number of "non-zero coefficients". Grey bars are coefficients that became zero.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

d = load_diabetes(as_frame=True)
X = StandardScaler().fit_transform(d.data); y = d.target
table = {}
for a in [0.01, 1, 5, 20, 50]:
    table[f"alpha={a}"] = Lasso(alpha=a).fit(X, y).coef_
coef = pd.DataFrame(table, index=d.data.columns).round(1)
print(coef)
print("Non-zero per alpha:", (coef != 0).sum().to_dict())



## Lasso or Ridge?
| Situation | Better choice |
|---|---|
| You think only a few features really matter | Lasso |
| All features have a small effect | Ridge |
| Strongly correlated features | Ridge or Elastic Net (Lasso picks one at random) |
| More features than samples | Lasso picks at most n features; Elastic Net is better |



In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = load_diabetes(as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(d.data, d.target, test_size=0.25, random_state=0)
m = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X_tr, y_tr)
print("Chosen alpha:", round(m[-1].alpha_, 3))
print("Kept features:", list(d.data.columns[m[-1].coef_ != 0]))
print("Test R2:", round(m.score(X_te, y_te), 3))

### Exercise

Write soft(rho, alpha) and reproduce the table in the lesson.

In [ ]:
# Your answer here

### Exercise

Add 50 useless random features to the diabetes data. How many does Lasso keep?

In [ ]:
# Your answer here